In [ ]:
# do install if needed. for example for collab runs
#!pip install -r decision-making-exam/requirements.txt


## Data Loading and Preprocessing

The data is recoded to have 8 arms based on the displayed amounts. The original 6-choice task had different stakes (low, medium, high). These can optionally be collapsed into 8 unique arms.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt


# Configuration
n_arms = 6  # 6 or 8 are valid. 8 will recode data based on displayed amounts

# Test mode: set to number of subjects per group for quick testing, or None for full data
test_n_per_group = 5  # Set to None for full dataset


df = pd.read_csv("../data/decmakdata.csv")

df.sample(10)

In [ ]:
df = pd.read_csv("../data/decmakdata.csv")

# Keep only relevant columns
df = df[["phase", "amount", "outcome", "choice", "condition", "group", "subject"]]

# Filter for experiment only, and win/loss outcomes to avoid signal trials and missed RTs
df = df.loc[df["phase"] == "experi"]
df = df.loc[df["outcome"].isin(["win", "loss"])]

# Subset subjects for testing if test_n_per_group is set
if test_n_per_group is not None:
    stop_subjects = df[df['group'] == 'bSTOP']['subject'].unique()[:test_n_per_group]
    double_subjects = df[df['group'] == 'bDOUBLE']['subject'].unique()[:test_n_per_group]
    selected_subjects = np.concatenate([stop_subjects, double_subjects])
    df = df[df['subject'].isin(selected_subjects)]
    print(f"TEST MODE: Using {test_n_per_group} subjects per group ({len(selected_subjects)} total)")

# Code new "displayed amount" column
# On losses, they lost half of the displayed amount; on wins they gained the displayed amount
df["displayed_amount"] = df["amount"].where(df["amount"] > 0, df["amount"] * -2)

# Shift rewards to be non-negative for the utility function
df["reward_scaled"] = (df["amount"] - df["amount"].min())

# Code new arm from displayed amount column (0 is highest outcome arm, 7 is lowest)
mapping = {value: i for i, value in enumerate(sorted(df["displayed_amount"].unique(), reverse=True))}
df["arm"] = df["displayed_amount"].map(mapping)

print(f"Number of subjects: {df['subject'].nunique()}")
print(f"Trials per subject: ~{len(df) // df['subject'].nunique()}")
print(f"\nGroup distribution:")
print(df.groupby('subject')['group'].first().value_counts())
print(f"\nArm mapping (displayed amount -> arm index):")
print(mapping)

df.sample(10)

In [ ]:
# Prepare data arrays for each subject
subj_arrays = []

for subj in sorted(df["subject"].unique()):
    subj_df = df.loc[df["subject"] == subj].iloc[:202]  # First 202 trials

    #get choice depeing on 6/8 arm config
    if n_arms == 8:
        choice_array = subj_df["arm"].to_numpy()
    else:
        choice_array = subj_df["choice"].to_numpy() - 1  # Convert to 0-indexed

    #get reward and stopcition
    reward_array = subj_df["amount"].to_numpy()
    stop_condition = subj_df["group"].iloc[0] == "bSTOP"
    # Indicates whether a load block was presented on each trial
    load_block_array = (subj_df["condition"] == "load").to_numpy()

    subj_arrays.append({
        "subject": subj,
        "choice_array": choice_array,
        "reward_array": reward_array,
        "stop_condition": stop_condition,
        "load_block_array": load_block_array
    })

# Setup data for PVL fit
choices = np.column_stack([d["choice_array"] for d in subj_arrays])
rewards = np.column_stack([d["reward_array"] for d in subj_arrays])
# Shape: (n_trials, n_subjects) - whether each subject was in a load block on each trial
load_blocks = np.column_stack([d["load_block_array"] for d in subj_arrays])
stop_condition = np.array([d["stop_condition"] for d in subj_arrays])
n_subjects = len(stop_condition)

print(f"Data shape: {choices.shape[0]} trials x {n_subjects} subjects")
print(f"Stop condition subjects: {stop_condition.sum()} / {n_subjects}")
print(f"Load block trials: {load_blocks.sum()} / {load_blocks.size}")
print(f"Reward range: [{rewards.min():.1f}, {rewards.max():.1f}]")

## Model Fitting

setup sampling below

In [ ]:
import sys
sys.path.insert(0, '..')  # Add parent dir for imports

import arviz as az
from numpyro.infer import NUTS, MCMC, Predictive
from pvl_delta import pvl_delta_model
import jax
import jax.numpy as jnp

# Set number of CPU cores for parallel chains
import numpyro
numpyro.set_host_device_count(8)

# MCMC configuration
NUM_WARMUP = 4000
NUM_SAMPLES = 2000
NUM_CHAINS = 8

print(f"Running MCMC with {NUM_CHAINS} chains, {NUM_WARMUP} warmup, {NUM_SAMPLES} samples each")
print(f"Total posterior samples: {NUM_CHAINS * NUM_SAMPLES}")

In [ ]:
from pathlib import Path

# Build filename that captures all relevant settings
test_suffix = f"_test{test_n_per_group}" if test_n_per_group else ""
filename = Path(f"../fitted_models/inference_data_{n_arms}arms_{NUM_CHAINS}chains_{NUM_SAMPLES}samples_{NUM_WARMUP}warmup{test_suffix}.nc")

# Ensure fitted_models directory exists
filename.parent.mkdir(parents=True, exist_ok=True)

if filename.exists():
    print(f"{filename.name} exists. Skipping MCMC and loading it")
    idata = az.from_netcdf(filename)
else:
    print(f"No matching fit found. Running MCMC...")
    print(f"  Arms: {n_arms}, Chains: {NUM_CHAINS}, Samples: {NUM_SAMPLES}, Warmup: {NUM_WARMUP}")
    inference_key = jax.random.key(42)
    nuts_kernel = NUTS(pvl_delta_model, target_accept_prob=0.9)
    mcmc = MCMC(nuts_kernel, num_samples=NUM_SAMPLES, num_warmup=NUM_WARMUP, num_chains=NUM_CHAINS)

    mcmc.run(
        inference_key,
        choices=choices,
        rewards=rewards,
        load_blocks=load_blocks,
        stop_condition=stop_condition,
        n_arms=n_arms,
        n_subjects=n_subjects,
    )

    # Convert to ArviZ InferenceData
    idata = az.from_numpyro(mcmc)
    # Save it
    az.to_netcdf(idata, filename)
    print(f"Saved to {filename.name}")

## Model Diagnostics


In [ ]:
# Print MCMC summary
#mcmc.print_summary() #didnt seem to work wiht loading idata object

In [ ]:
# Key population-level parameters to check
pop_params = [
    'lr_loc', 'lr_scale', 'stop_lr_effect',
    'inv_t_loc', 'inv_t_scale', 'stop_inv_t_effect',
    'u_shape_loc', 'u_shape_scale', 'stop_u_shape_effect',
    'u_aversion_loc', 'u_aversion_scale', 'stop_u_aversion_effect'
]

# Compute diagnostics
summary = az.summary(idata, var_names=pop_params, hdi_prob=0.94)
print("=" * 80)
print("POPULATION-LEVEL PARAMETER SUMMARY")
print("=" * 80)
print(summary.to_string())

# Check for issues
print("\n" + "=" * 80)
print("DIAGNOSTIC CHECKS")
print("=" * 80)

# R-hat check
max_rhat = summary['r_hat'].max()
print(f"Max R-hat: {max_rhat:.4f} {'✓' if max_rhat < 1.01 else '✗ WARNING: Poor convergence'}")

# ESS check
min_ess = min(summary['ess_bulk'].min(), summary['ess_tail'].min())
print(f"Min ESS:   {min_ess:.0f} {'✓' if min_ess > 400 else '✗ WARNING: Low effective samples'}")

# Divergences
n_divergent = idata.sample_stats.diverging.sum().item()
print(f"Divergences: {n_divergent} {'✓' if n_divergent == 0 else '⚠ WARNING: Check model'}")

In [ ]:
# Trace plots for key parameters
effect_params = ['stop_lr_effect', 'stop_inv_t_effect', 'stop_u_shape_effect', 'stop_u_aversion_effect']
az.plot_trace(idata, var_names=effect_params, figsize=(12, 8))
plt.suptitle('Trace Plots: Stop Condition Effects', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Trace plots for population location parameters
loc_params = ['lr_loc', 'inv_t_loc', 'u_shape_loc', 'u_aversion_loc']
az.plot_trace(idata, var_names=loc_params, figsize=(12, 8))
plt.suptitle('Trace Plots: Population Location Parameters', y=1.02)
plt.tight_layout()
plt.show()

## Results: Stop Condition Effects

These are the key parameters for testing our hypotheses. A credible interval that excludes 0 suggests a reliable effect.

In [ ]:
# Forest plot of stop condition effects
fig, ax = plt.subplots(figsize=(10, 5))
az.plot_forest(
    idata, 
    var_names=effect_params,
    combined=True,
    hdi_prob=0.94,
    ax=ax
)
ax.axvline(0, color='red', linestyle='--', alpha=0.7, label='Null effect')
ax.set_title('Stop Condition Effects (94% HDI)', fontsize=14)
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Posterior distributions of stop effects
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

effect_labels = {
    'stop_lr_effect': 'Learning Rate Effect\n(+ = faster learning in stop)',
    'stop_inv_t_effect': 'Inverse Temperature Effect\n(+ = more consistent in stop)',
    'stop_u_shape_effect': 'Utility Shape Effect\n(+ = less risk averse in stop)',
    'stop_u_aversion_effect': 'Loss Aversion Effect\n(+ = more loss averse in stop)'
}

for ax, param in zip(axes.flat, effect_params):
    samples = idata.posterior[param].values.flatten()
    
    # Plot posterior
    az.plot_posterior(
        idata, 
        var_names=[param], 
        hdi_prob=0.94,
        ax=ax,
        ref_val=0
    )
    ax.set_title(effect_labels[param], fontsize=11)
    
    # Calculate probability of effect > 0
    prob_positive = (samples > 0).mean()
    ax.text(0.02, 0.98, f'P(effect > 0) = {prob_positive:.3f}', 
            transform=ax.transAxes, fontsize=10, va='top')

plt.suptitle('Posterior Distributions of Stop Condition Effects', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Detailed summary of stop effects
print("=" * 80)
print("STOP CONDITION EFFECTS: DETAILED SUMMARY")
print("=" * 80)

for param in effect_params:
    samples = idata.posterior[param].values.flatten()
    mean = samples.mean()
    std = samples.std()
    hdi = az.hdi(samples, hdi_prob=0.94)
    prob_pos = (samples > 0).mean()
    prob_neg = (samples < 0).mean()
    
    print(f"\n{param}:")
    print(f"  Mean: {mean:.4f} (SD: {std:.4f})")
    print(f"  94% HDI: [{hdi[0]:.4f}, {hdi[1]:.4f}]")
    print(f"  P(effect > 0): {prob_pos:.3f}")
    print(f"  P(effect < 0): {prob_neg:.3f}")
    
    # Interpretation
    if hdi[0] > 0:
        print(f"  --> Credible POSITIVE effect")
    elif hdi[1] < 0:
        print(f"  --> Credible NEGATIVE effect")
    else:
        print(f"  --> Effect includes 0 (inconclusive)")

## Subject-Level Parameters

Comparing parameter distributions between Stop and Double-Response groups.

In [ ]:
# Get posterior means for subject-level parameters
subject_params = ['lr', 'inv_t', 'u_shape', 'u_aversion']

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

for ax, param in zip(axes.flat, subject_params):
    # Get posterior samples for this parameter (subjects on last axis)
    samples = idata.posterior[param].values  # shape: (chains, draws, subjects)
    mean_per_subject = samples.mean(axis=(0, 1))  # Average over chains and draws
    
    stop_vals = mean_per_subject[stop_condition]
    double_vals = mean_per_subject[~stop_condition]
    
    # Box plot comparison
    bp = ax.boxplot([double_vals, stop_vals], labels=['Double-Response', 'Stop'], patch_artist=True)
    bp['boxes'][0].set_facecolor('lightblue')
    bp['boxes'][1].set_facecolor('lightcoral')
    
    ax.set_ylabel(param)
    ax.set_title(f'{param} by Group')
    
    # Add individual points
    ax.scatter(np.ones(len(double_vals)) + np.random.randn(len(double_vals))*0.05, 
               double_vals, alpha=0.5, color='blue', s=20)
    ax.scatter(np.ones(len(stop_vals))*2 + np.random.randn(len(stop_vals))*0.05, 
               stop_vals, alpha=0.5, color='red', s=20)
    
    # Print means
    ax.text(0.02, 0.98, f'Double-Resp: {double_vals.mean():.3f}\nStop: {stop_vals.mean():.3f}',
            transform=ax.transAxes, va='top', fontsize=10)

plt.suptitle('Subject-Level Parameter Estimates by Group', fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

## Posterior Predictive Check

We check if the model can reproduce the observed choice patterns.

In [ ]:
# Generate posterior predictive samples
predictive = Predictive(pvl_delta_model, mcmc.get_samples())
ppc_key = jax.random.key(123)

posterior_predictive = predictive(
    ppc_key,
    choices=choices,
    rewards=rewards,
    load_blocks=load_blocks,
    stop_condition=stop_condition,
    n_arms=n_arms,
    n_subjects=n_subjects,
)

# Add to idata
idata.extend(az.from_numpyro(posterior_predictive=posterior_predictive))

In [ ]:
# Compare predicted vs observed choice distributions
pred_choices = idata.posterior_predictive['obs'].values  # (chains, draws, trials, subjects)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, (group_name, mask) in zip(axes, [('Stop', stop_condition), ('Double-Response', ~stop_condition)]):
    # Observed
    obs_flat = choices[:, mask].flatten()
    
    # Predicted (take mean across chains and draws)
    pred_flat = pred_choices[:, :, :, mask].flatten()
    
    bins = np.arange(n_arms + 1) - 0.5
    ax.hist(obs_flat, bins=bins, alpha=0.6, label='Observed', density=True, color='blue')
    ax.hist(pred_flat, bins=bins, alpha=0.4, label='Predicted', density=True, color='orange')
    
    ax.set_xlabel('Choice (arm)')
    ax.set_ylabel('Density')
    ax.set_title(f'{group_name} Group')
    ax.legend()

plt.suptitle('Posterior Predictive Check: Choice Distributions', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Calculate predictive accuracy
from sklearn.metrics import accuracy_score

# Mode prediction (most common predicted choice per trial)
pred_mode = np.apply_along_axis(
    lambda x: np.bincount(x.astype(int), minlength=n_arms).argmax(),
    axis=(0, 1),
    arr=pred_choices
)

# Overall accuracy
overall_acc = accuracy_score(choices.flatten(), pred_mode.flatten())

# By group
stop_acc = accuracy_score(choices[:, stop_condition].flatten(), 
                          pred_mode[:, stop_condition].flatten())
double_acc = accuracy_score(choices[:, ~stop_condition].flatten(), 
                            pred_mode[:, ~stop_condition].flatten())

# Chance level
chance = 1 / n_arms

print(f"Predictive Accuracy (mode prediction):")
print(f"  Overall: {overall_acc:.3f} (chance: {chance:.3f})")
print(f"  Stop group: {stop_acc:.3f}")
print(f"  Double-Response group: {double_acc:.3f}")

## Interpretation and Conclusions

Based on the project statement, we evaluate:

1. **Primary hypothesis (Stevens et al.)**: Does motor inhibition change how participants value risky vs safe options?
   - Look at `stop_u_shape_effect` and `stop_u_aversion_effect`

2. **Alternative hypothesis 1**: Is the effect due to improved probability learning?
   - Look at `stop_lr_effect`

3. **Alternative hypothesis 2**: Is the effect due to more consistent behavior?
   - Look at `stop_inv_t_effect`

In [ ]:
print("=" * 80)
print("INTERPRETATION OF RESULTS")
print("=" * 80)

# Get HDIs for each effect
effects = {}
for param in effect_params:
    samples = idata.posterior[param].values.flatten()
    effects[param] = {
        'mean': samples.mean(),
        'hdi': az.hdi(samples, hdi_prob=0.94),
        'prob_pos': (samples > 0).mean(),
        'prob_neg': (samples < 0).mean()
    }

print("\n1. PRIMARY HYPOTHESIS (Stevens et al.): Utility/Value Changes")
print("-" * 60)

# U-shape effect
e = effects['stop_u_shape_effect']
print(f"\n  Utility Shape Effect:")
print(f"    Mean: {e['mean']:.4f}, 94% HDI: [{e['hdi'][0]:.4f}, {e['hdi'][1]:.4f}]")
if e['hdi'][0] > 0:
    print(f"    --> Stop group shows HIGHER utility curvature (less risk averse)")
elif e['hdi'][1] < 0:
    print(f"    --> Stop group shows LOWER utility curvature (more risk averse)")
else:
    print(f"    --> No credible effect on utility curvature")

# U-aversion effect  
e = effects['stop_u_aversion_effect']
print(f"\n  Loss Aversion Effect:")
print(f"    Mean: {e['mean']:.4f}, 94% HDI: [{e['hdi'][0]:.4f}, {e['hdi'][1]:.4f}]")
if e['hdi'][0] > 0:
    print(f"    --> Stop group shows HIGHER loss aversion")
elif e['hdi'][1] < 0:
    print(f"    --> Stop group shows LOWER loss aversion")
else:
    print(f"    --> No credible effect on loss aversion")

print("\n2. ALTERNATIVE HYPOTHESIS 1: Probability Learning")
print("-" * 60)
e = effects['stop_lr_effect']
print(f"\n  Learning Rate Effect:")
print(f"    Mean: {e['mean']:.4f}, 94% HDI: [{e['hdi'][0]:.4f}, {e['hdi'][1]:.4f}]")
if e['hdi'][0] > 0:
    print(f"    --> Stop group shows FASTER learning")
elif e['hdi'][1] < 0:
    print(f"    --> Stop group shows SLOWER learning")
else:
    print(f"    --> No credible effect on learning rate")

print("\n3. ALTERNATIVE HYPOTHESIS 2: Behavioral Consistency")
print("-" * 60)
e = effects['stop_inv_t_effect']
print(f"\n  Inverse Temperature Effect:")
print(f"    Mean: {e['mean']:.4f}, 94% HDI: [{e['hdi'][0]:.4f}, {e['hdi'][1]:.4f}]")
if e['hdi'][0] > 0:
    print(f"    --> Stop group shows MORE CONSISTENT choice behavior")
elif e['hdi'][1] < 0:
    print(f"    --> Stop group shows LESS CONSISTENT choice behavior")
else:
    print(f"    --> No credible effect on behavioral consistency")

print("\n" + "=" * 80)
print("OVERALL CONCLUSION")
print("=" * 80)

In [ ]:
# Save inference data for later use
idata.to_netcdf('../fitted_models/fit_results.nc')
print("Results saved to ../fitted_models/fit_results.nc")